In [2]:
# 使用GDAL读取HDF4数据
from osgeo import gdal

ds_in = gdal.Open('/Users/tanzhenyu/Dataware/GeoPy/07/example.hdf')
# 返回结果是一个list，list中的每个元素是一个tuple，每个tuple中包含了对数据集的路径，元数据等的描述信息
# tuple中的第一个元素描述的是数据子集的全路径
subdatasets = ds_in.GetSubDatasets()

band1 = gdal.Open(subdatasets[0][0])  # 取出第1个数据子集（MODIS反射率产品的第一个波段）
arr_b1 = band1.ReadAsArray()  # 将数据集中的数据转为ndarray
print(arr_b1.shape)

# 创建输出数据集，转为GeoTIFF进行写入
# 但是这样写入的数据是没有空间参考信息的，若要将投影信息写入还需要额外处理
fn_out = '/Users/tanzhenyu/Dataware/GeoPy/07/sur_refl_b01.tif'
driver = gdal.GetDriverByName('GTiff')
ds_out = driver.CreateCopy(fn_out, band1)
ds_out.GetRasterBand(1).WriteArray(arr_b1)
ds_out.FlushCache()

# 关闭数据集
del ds_out
del ds_in

(2400, 2400)


In [6]:
import rasterio
from rasterio.errors import NotGeoreferencedWarning
import warnings

fn_in = '/Users/tanzhenyu/Dataware/GeoPy/07/example.hdf'
# 读取HDF以及NetCDF数据的时候会提示没有空间参考信息，我们可以在代码中忽略这个警告，当然也可以不处理
with warnings.catch_warnings():
    warnings.filterwarnings('ignore', category=NotGeoreferencedWarning)
    with rasterio.open(fn_in) as ds:
        for name in ds.subdatasets:
            print(name)
            
        # 读取第一个波段的像素信息为一个NDArray
        with rasterio.open(ds.subdatasets[0]) as ds:
            print(ds.profile) # 输出该数据的元数据信息
            data = ds.read() # 将数据读取为NumPy的NDArray
            print(data.shape) # 输出图像尺寸


HDF4_EOS:EOS_GRID:/Users/tanzhenyu/Dataware/GeoPy/07/example.hdf:MOD_Grid_500m_Surface_Reflectance:sur_refl_b01
HDF4_EOS:EOS_GRID:/Users/tanzhenyu/Dataware/GeoPy/07/example.hdf:MOD_Grid_500m_Surface_Reflectance:sur_refl_vzen
HDF4_EOS:EOS_GRID:/Users/tanzhenyu/Dataware/GeoPy/07/example.hdf:MOD_Grid_500m_Surface_Reflectance:sur_refl_raz
HDF4_EOS:EOS_GRID:/Users/tanzhenyu/Dataware/GeoPy/07/example.hdf:MOD_Grid_500m_Surface_Reflectance:sur_refl_state_500m
HDF4_EOS:EOS_GRID:/Users/tanzhenyu/Dataware/GeoPy/07/example.hdf:MOD_Grid_500m_Surface_Reflectance:sur_refl_day_of_year
HDF4_EOS:EOS_GRID:/Users/tanzhenyu/Dataware/GeoPy/07/example.hdf:MOD_Grid_500m_Surface_Reflectance:sur_refl_b02
HDF4_EOS:EOS_GRID:/Users/tanzhenyu/Dataware/GeoPy/07/example.hdf:MOD_Grid_500m_Surface_Reflectance:sur_refl_b03
HDF4_EOS:EOS_GRID:/Users/tanzhenyu/Dataware/GeoPy/07/example.hdf:MOD_Grid_500m_Surface_Reflectance:sur_refl_b04
HDF4_EOS:EOS_GRID:/Users/tanzhenyu/Dataware/GeoPy/07/example.hdf:MOD_Grid_500m_Surface_R

In [16]:
from netCDF4 import Dataset, Variable
import numpy as np
from numpy.ma import MaskedArray

fn_in = '/Users/tanzhenyu/Dataware/GeoPy/07/example.nc'
with Dataset(fn_in) as ds:
    print(ds.variables)
    print(ds.dimensions)
    data: Variable = ds.variables['LU_INDEX'] # 类型为Variable
    print(data.dimensions) # 查看数据的维度，这个是一个三维的数据
    print(data.shape) # 当然其Shape也是对应在三个维度上的数量
    # 使用[:]运算符可以将Variable转变为NumPy的类型为MaskedArray
    data: MaskedArray = ds.variables['LU_INDEX'][:] # 类型为MaskedArray
    # 因为第一个维度时间维只有一个时间节点，使用np.squeeze()方法可以将第一个维度去掉，将数据变为三维
    print(np.squeeze(data).shape) 

{'Times': <class 'netCDF4._netCDF4.Variable'>
|S1 Times(Time, DateStrLen)
unlimited dimensions: Time
current shape = (1, 19)
filling on, default _FillValue of   used, 'XLAT_M': <class 'netCDF4._netCDF4.Variable'>
float32 XLAT_M(Time, south_north, west_east)
    FieldType: 104
    MemoryOrder: XY 
    units: degrees latitude
    description: Latitude on mass grid
    stagger: M
    sr_x: 1
    sr_y: 1
unlimited dimensions: Time
current shape = (1, 45, 66)
filling on, default _FillValue of 9.969209968386869e+36 used, 'XLONG_M': <class 'netCDF4._netCDF4.Variable'>
float32 XLONG_M(Time, south_north, west_east)
    FieldType: 104
    MemoryOrder: XY 
    units: degrees longitude
    description: Longitude on mass grid
    stagger: M
    sr_x: 1
    sr_y: 1
unlimited dimensions: Time
current shape = (1, 45, 66)
filling on, default _FillValue of 9.969209968386869e+36 used, 'XLAT_V': <class 'netCDF4._netCDF4.Variable'>
float32 XLAT_V(Time, south_north_stag, west_east)
    FieldType: 104
    Me

In [26]:
from pyhdf.SD import SD, SDC, SDS
import numpy as np

fn_in = '/Users/tanzhenyu/Dataware/GeoPy/07/example.hdf'
# 以只读模式打开HDF4文件，读取元数据信息
# SD类代表HDF数据集
ds = SD(fn_in, SDC.READ)
# 读取该HDF4文件中包含的子数据集
subdatasets = ds.datasets()
for idx, name in enumerate(subdatasets.keys()):
    print(f"{idx}: {name}")

# 读取第一个波段的反射率数据
name = list(subdatasets.keys())[0]
ds_bnd: SDS = ds.select(name)
data: np.ndarray = ds_bnd.get() # 将HDF4子数据集转为NumPy的NDarray
print(data.shape)

# 记得关闭数据集
ds.end()

0: sur_refl_b01
1: sur_refl_b02
2: sur_refl_b03
3: sur_refl_b04
4: sur_refl_b05
5: sur_refl_b06
6: sur_refl_b07
7: sur_refl_qc_500m
8: sur_refl_szen
9: sur_refl_vzen
10: sur_refl_raz
11: sur_refl_state_500m
12: sur_refl_day_of_year


KeyboardInterrupt: 

In [11]:
import h5py
from h5py import Dataset


fn_in = '/Users/tanzhenyu/Dataware/GeoPy/07/example.h5'
with h5py.File(fn_in, 'r') as ds:
    # 注意这里的ds变量的类型为h5py中的File
    # 打印元数据信息，类似于我们使用gdalinfo命令查看数据信息
    ds.visititems(lambda name, obj: print(f'{name}: {obj}'))
    # 使用波段名称读取对应波段的数据（类型为h5py的Dataset）
    band: Dataset = ds['bands/Rw555']
    # 使用[:]操作符将Dataset转为NumPy的NDArray
    data = band[:]
    print(data.shape)

bands: <HDF5 group "/bands" (20 members)>
bands/Rgli: <HDF5 dataset "Rgli": shape (189, 162), type "<f4">
bands/Rnir: <HDF5 dataset "Rnir": shape (189, 162), type "<f4">
bands/Rw412: <HDF5 dataset "Rw412": shape (189, 162), type "<f4">
bands/Rw443: <HDF5 dataset "Rw443": shape (189, 162), type "<f4">
bands/Rw469: <HDF5 dataset "Rw469": shape (189, 162), type "<f4">
bands/Rw488: <HDF5 dataset "Rw488": shape (189, 162), type "<f4">
bands/Rw531: <HDF5 dataset "Rw531": shape (189, 162), type "<f4">
bands/Rw547: <HDF5 dataset "Rw547": shape (189, 162), type "<f4">
bands/Rw555: <HDF5 dataset "Rw555": shape (189, 162), type "<f4">
bands/Rw645: <HDF5 dataset "Rw645": shape (189, 162), type "<f4">
bands/Rw667: <HDF5 dataset "Rw667": shape (189, 162), type "<f4">
bands/Rw678: <HDF5 dataset "Rw678": shape (189, 162), type "<f4">
bands/Rw748: <HDF5 dataset "Rw748": shape (189, 162), type "<f4">
bands/Rw869: <HDF5 dataset "Rw869": shape (189, 162), type "<f4">
bands/bitmask: <HDF5 dataset "bitmask"

In [24]:
import xarray as xr
from xarray import DataArray
from datetime import datetime

fn_in = '/Users/tanzhenyu/Dataware/GeoPy/07/example.nc'
# 变量ds的数据类型为xr的Dataset
with xr.open_dataset(fn_in) as ds:
    print(ds.data_vars) # 该数据包含哪些变量
    print(ds.coords) # 该数据支持的坐标系（对于地理数据我们有经纬度和时间的具体范围）
    print(ds.dims) # 维度即为坐标系的名称
    
    data: DataArray = ds['tp']
    print(data.coords)
    array = data.isel(hybas_id=0)
    print(array.shape)
    array = data.isel(hybas_id=slice(1, 5))
    print(array.shape)
    array = data.sel(time=datetime(2000, 1, 1))
    print(array.shape)
    array = data.sel(time=datetime(2000, 1, 1), hybas_id=slice('2120313970', '2120320920'))
    print(array.shape)
    

Data variables:
    licd     (hybas_id, time) float64 ...
    lmlt     (hybas_id, time) float64 ...
    skt      (hybas_id, time) float64 ...
    u10      (hybas_id, time) float64 ...
    v10      (hybas_id, time) float64 ...
    ssrd     (hybas_id, time) float64 ...
    tp       (hybas_id, time) float64 ...
Coordinates:
  * hybas_id  (hybas_id) object '2120313970' '2120316200' ... '2121091950' 'All'
  * time      (time) datetime64[ns] 1950-01-01 1950-01-02 ... 2023-12-31
Frozen({'hybas_id': 39, 'time': 27028})
Coordinates:
  * hybas_id  (hybas_id) object '2120313970' '2120316200' ... '2121091950' 'All'
  * time      (time) datetime64[ns] 1950-01-01 1950-01-02 ... 2023-12-31
(27028,)
(4, 27028)
(39,)
(22,)
